# Figure 4 — scaling the problem and the network

Block-sparse and dense networks from 4 to 512 hidden units, on multi-MNIST
problems of 2 to 32 tasks, 5 seeds per cell. The permutation rate is scaled
with the task count so that each individual task is re-permuted once every
2000 steps on average, which keeps the amount of non-stationarity *per task*
fixed as the problem grows.

The cost of dense connectivity grows sharply with the task count. From 8 tasks
upward, no dense network of any size reaches what block-sparse reaches at its
best: at 32 tasks block-sparse peaks near 86% while the best dense network of
any size stops at about 31%. Dense performance also degrades past a certain
size at every task count, so a network cannot scale its way out of a bad
credit assignment problem.

Produced by `experiments/sweeps/03_task_scaling/`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from multi_mnist.analysis import (
    asymptotic_metric, filter_sweep, get_color_palette, load_export,
    normalize_columns, plot_sensitivity, save_fig, set_style,
)

%matplotlib inline


In [ ]:
# The Comet project the published runs live in. Change it if you
# re-run the sweeps into your own project.
PROJECT = 'paper-weight-pruning-scaling-sweep'

# This repository's sweep names, plus the names the published runs used.
SWEEPS = {
    'Block-Sparse': ['04_block_sparse_task_scaling',
                     '00_block_sparse_n_task_scaling_lr_5seed'],
    'Dense':        ['04_dense_task_scaling',
                     '00_dense_n_task_scaling_lr_5seed'],
}

CELL_COLS = ['task.n_tasks', 'model.initial_hidden_units', 'optimizer.learning_rate']
TASK_COUNTS = [2, 4, 8, 16, 32]
HIDDEN_UNITS = [4, 8, 16, 32, 64, 128, 256, 512]


In [ ]:
# Run once to fetch from Comet, then leave commented out.
# from multi_mnist.analysis import download_project
# download_project(PROJECT, output_dir='data')

cfg_all, run_all = load_export(PROJECT, data_dir='data')
cfg_all = normalize_columns(cfg_all)
print(f'{len(cfg_all)} trials, {len(run_all)} metric rows')


## Best step-size per (task count, size) cell

Each cell of the figure is that configuration at its own best step-size, so
the comparison is not confounded by one method tolerating a larger step-size
than another. Block-sparse cells with fewer hidden units than tasks are
missing by construction — it needs at least one unit per task.


In [ ]:
def best_per_cell(method):
    cfg, run = filter_sweep(cfg_all, run_all, SWEEPS[method], cell_cols=CELL_COLS)
    asym = asymptotic_metric(
        cfg, run, 'accuracy', ['task.n_tasks', 'model.initial_hidden_units'],
        tail_frac=0.10)
    idx = asym.groupby(['task.n_tasks', 'model.initial_hidden_units'])['_metric'] \
              .idxmax().dropna()
    return asym.loc[idx.astype(int)]


best = {m: best_per_cell(m) for m in SWEEPS}
for method, df in best.items():
    piv = df.pivot(index='task.n_tasks', columns='model.initial_hidden_units',
                   values='_metric')
    print(f'\n{method} — asymptotic accuracy (rows = tasks, cols = hidden units)')
    print(piv.round(3).to_string())


## Figure 4 — accuracy vs network size, one line per task count

In [ ]:
def plot_panel(method, savename):
    df = best[method]
    colors = sns.color_palette('viridis', n_colors=len(TASK_COUNTS))

    set_style('2-col')
    fig, ax = plt.subplots()
    for n_tasks, color in zip(TASK_COUNTS, colors):
        sub = df[df['task.n_tasks'] == n_tasks].sort_values('model.initial_hidden_units')
        ax.plot(sub['model.initial_hidden_units'], sub['_metric'], '-o',
                color=color, label=str(n_tasks))
    ax.set_xscale('log', base=2)
    ax.set_xticks(HIDDEN_UNITS)
    ax.set_xticklabels([f'$2^{{{int(np.log2(h))}}}$' for h in HIDDEN_UNITS])
    ax.set_xlabel('Hidden Unit Count'); ax.set_ylabel('Accuracy')
    ax.set_title(method); ax.set_ylim(0, 1.0); ax.grid(True, alpha=0.4)
    ax.legend(title='Task Count', loc='lower right')
    save_fig(savename, fig_dir='../figures/generated')
    plt.show()


plot_panel('Block-Sparse', 'fig4_block_sparse_task_scaling')
plot_panel('Dense', 'fig4_dense_task_scaling')


## Block-sparse against dense at every size

For each task count: the smallest block-sparse network that can represent the
problem at all (one hidden unit per task), block-sparse at its best size, and
dense at its best size.

The smallest block-sparse network is not the interesting comparison — one unit
per task is too little capacity to do anything, and dense beats it from 8
tasks up. The gap that matters is the third column against the second.

In [ ]:
rows = []
for n_tasks in TASK_COUNTS:
    bs = best['Block-Sparse']
    bs_n = bs[bs['task.n_tasks'] == n_tasks]
    de = best['Dense']
    de_n = de[de['task.n_tasks'] == n_tasks]
    if bs_n.empty or de_n.empty:
        continue
    smallest = bs_n.loc[bs_n['model.initial_hidden_units'].idxmin()]
    rows.append({
        'tasks': n_tasks,
        'smallest block-sparse': f"{smallest['_metric']:.3f} "
                                 f"(h={int(smallest['model.initial_hidden_units'])})",
        'best block-sparse': f'{bs_n["_metric"].max():.3f}',
        'best dense (any size)': f'{de_n["_metric"].max():.3f}',
    })
print(pd.DataFrame(rows).to_string(index=False))
